In [38]:
#import sys
#!{sys.executable} -m pip install import_ipynb

In [39]:
#import import_ipynb
#from BPDRR_crew_allocation import allocate_crews
from pprint import pprint
import random
import pandas as pd
import numpy as np

In [40]:
def allocate_crews(reparations, dmatrix, indexes, n_teams):
    missing_repairs = set(indexes) - set(reparations)
    if missing_repairs:
        raise KeyError(f"IDs missing from reparations: {sorted(missing_repairs)}")

    missing_matrix = set(indexes) - set(dmatrix.index)
    if missing_matrix:
        raise KeyError(f"IDs missing from travel-time matrix: {sorted(missing_matrix)}")

    # existing allocation logic...
    """
    allocate_crews(reparations, dmatrix, indexes, n_teams):
    
    Description
       This is essentially a greedy load-balancing problem: 
       process jobs in the order given by indexes, and always assign the next job 
       to the crew with the smallest accumulated repair time.
    
    Input Parameters
    -----------------
    reparations : dict
        Dictionary {id: repair_time}
    dmatrix : dict
        Dictionary {id_i: {id_j: travel_time}}
    indexes : list
        List of ids indicating the allocation order, must have the same length of the keys of 'reparations'
    n_teams : int
        Number of crews, how many crews are you sending in the field to repair pipes

    Output / Returns
    ----------------
    dictionary with the pipe ids and the times of interventions for each crew. 
    dict { 'crew_1': { 'pipe_ids': [...], 'time_total': ...},
           'crew_2': { 'pipe_ids': [...], 'time_total': ...}, 
           ...
         }

    Developed by : Mario Castro-Gama, ir. MSc. PhD
    Last update  : 2026-05-20
                   2026-06-15, added travel time constant
                   2026-07-21, added travel tiem as function of 'dmatrix' distance matrix
    
    """

    # if each crew key is a string
    # crews = { f"crew_{i+1}": {'pipe_ids':  [], 
    #                           'time_k':    [], 
    #                           'time_t':    [], 
    #                           'time_0':    [],
    #                           'time_1':    [],
    #                           'time_total': 0,
    #                          } for i in range(n_teams)}

    # if each crew key is an int (starting at 1)
    crews = { i+1: {'pipe_ids':  [], 
                    'time_repair':    [], 
                    'time_travel':   [],
                    'time_0':    [],
                    'time_1':    [],
                    'time_total': 0,
                   } for i in range(n_teams)}

    nrep = len(reparations)

    print('')
    print('organize distances for each crew')
    new_controls = []
    for idx in indexes:
        if idx not in reparations:
            raise KeyError(f"Pipe ID '{idx}' not found in reparations")

        # Find the crew with the minimum accumulated time to allocate the next reparation
        # first available gets chosen
        crew_curr = min(crews, key = lambda c: crews[c]['time_total'])

        # current reparation time
        repair_time = reparations[idx]['t_r']

        # find the travel time between this pipe and the previous one
        if crews[crew_curr]['pipe_ids']==[]:
            travel_time = 0.5  # no previous pipe so give it 30 minutes
        else:
            # This is estimated from distance matrix 'dmatrix', that matrix is square and static for each damage scenario
            pipe_prev = crews[crew_curr]['pipe_ids'][-1]
            travel_time = dmatrix[pipe_prev][idx]
            print('Crew '+str(crew_curr)+', from '+pipe_prev+'-to-'+idx+' : '+str(travel_time))
        
        # Assign the reparation to each crew
        crews[crew_curr]['pipe_ids'].append(idx)
        crews[crew_curr]['time_repair'].append(repair_time)
        crews[crew_curr]['time_travel'].append(travel_time)
        crews[crew_curr]['time_0'].append(travel_time + crews[crew_curr]['time_total'])
        crews[crew_curr]['time_1'].append(repair_time + crews[crew_curr]['time_0'][-1])
        crews[crew_curr]['time_total'] += repair_time + travel_time

        new_controls.append(f"; Crew {crew_curr} - Pipe {idx}\n")
        new_controls.append(f"LINK {idx}_A CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<A> when arriving to the location
        new_controls.append(f"LINK {idx}_B CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<B> when arriving to the location
        new_controls.append(f"LINK {idx} OPEN AT TIME {crews[crew_curr]['time_1'][-1]}\n")     # Open Pipe_id only after time of reparation
        
    return crews, new_controls

In [41]:
# Damage scenario selection
ds_sel = 'DS1'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():
    time_reparation[str(row["Pipe ID"])] = {
        "t_r": float(row["fix time (hours)"])
    }

print(f"{len(time_reparation)} repairs loaded.")
print(list(time_reparation.items())[5:10])

125 repairs loaded.
[('5698', {'t_r': 7.039878523352293}), ('3045', {'t_r': 5.724455894848762}), ('338', {'t_r': 5.724455894848762}), ('5837', {'t_r': 5.724455894848762}), ('3391', {'t_r': 4.276855708207498})]


## Example 1
This example uses a dmatrix that uses ramdom values

In [42]:
#n_teams = 3

# Pipe IDs from the reparations dictionary
#pipe_ids = list(time_reparation.keys())

# Number of repairs
#n_rep = len(pipe_ids)

# Random travel times between 0 and 2 hours
#random_matrix = np.random.uniform(0, 2, (n_rep, n_rep))

# Travel from a pipe to itself is zero
#np.fill_diagonal(random_matrix, 0)

# Create DataFrame with pipe IDs
#dmatrix_df = pd.DataFrame(
#    random_matrix,
#    index=pipe_ids,
#    columns=pipe_ids
#)

#dmatrix_df.head()

In [43]:
# generate a random permutation, one would expect to get this directly from the optimization (PYMOO)
#indexes = random.sample(pipe_ids, n_rep)
#print('Show permutation of reparations')
#print(indexes)
#print(len(indexes))

In [44]:
# apply the greedy allocation to the dataset
#crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
#print('')
#print('[CONTROLS]')
#pprint(new_controls)

In [45]:
#pd.DataFrame({"crews": new_controls}).to_excel(
#    "new_controls.xlsx",
#    index=False
#)

## Example 2
This example uses a dmatrix imported from the files generated by "BPDRR_travel_time_matrix_gen.ipynb"

In [46]:
# Number of crews
n_teams = 3

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [ ]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_excel(
    "TravelTime_matrices.xlsx",
    sheet_name= ds_sel,
    index_col=0
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,437,3602,2112,3562,5333,5698,3045,338,5837,3391,...,5453,5577,5793,5920,605,614,69,740,880,940
437,0.011744,0.299856,0.260351,0.200540,0.313448,0.241282,0.278857,0.149123,0.256378,0.222719,...,0.306900,0.331890,0.337652,0.347845,0.188247,0.163556,0.140375,0.173430,0.185285,0.186333
3602,0.299856,0.004956,0.235733,0.103863,0.613304,0.541138,0.070301,0.234137,0.556233,0.126296,...,0.606756,0.631746,0.637507,0.647701,0.241429,0.184273,0.252501,0.199663,0.213933,0.212928
2112,0.260351,0.235733,0.000877,0.136418,0.573799,0.501633,0.214735,0.213793,0.516729,0.158597,...,0.567251,0.592241,0.598003,0.608196,0.252917,0.222594,0.159604,0.237165,0.249956,0.250249
3562,0.200540,0.103863,0.136418,0.000697,0.513989,0.441822,0.088787,0.136039,0.456918,0.024964,...,0.507440,0.532431,0.538192,0.548385,0.143332,0.086176,0.153186,0.101566,0.115836,0.114831
5333,0.313448,0.613304,0.573799,0.513989,0.001241,0.076458,0.592305,0.462571,0.075214,0.536168,...,0.035420,0.018442,0.031372,0.044182,0.501695,0.477005,0.453823,0.486878,0.498734,0.499781


In [48]:
indexes = random.sample(pipe_ids, n_rep)
print('Show permutation of reparations')
print(indexes)
print(len(indexes))

Show permutation of reparations
['2597', '1763', '5135', '4757', '616', '3257', '1818', '3571', '3772', '1702', '840', '254', '1060', '5105', '4661', '4895', '5203', '3817', '5062', '1978', '5933', '6005', '5698', '4652', '3446', '3273', '210', '3038', '3045', '2298', '3693', '1176', '3101', '902', '3681', '3387', '1969', '3854', '4594', '5436', '605', '5333', '3404', '1701', '5793', '517', '1892', '69', '5577', '6065', '3187', '740', '32', '630', '191', '4082', '3497', '3124', '3545', '3572', '450', '3834', '610', '3897', '4319', '3640', '3884', '3602', '940', '5188', '3562', '1221', '47', '1952', '3331', '2704', '2112', '1241', '4518', '3391', '5497', '1175', '446', '5837', '5920', '880', '618', '5066', '4505', '776', '3144', '1595', '614', '4050', '2705', '2999', '3147', '4369', '2599', '5869', '1413', '620', '3895', '5453', '4805', '363', '3969', '4880', '1593', '58', '4889', '437', '1369', '3285', '338', '3342', '380', '5023', '5787', '38', '3950', '5067', '280', '2065', '1379']
1

In [49]:
# apply the greedy allocation to the dataset
crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)


organize distances for each crew
Crew 1, from 2597-to-4757 : 0.4137978048780487
Crew 2, from 1763-to-616 : 0.2400812195121951
Crew 3, from 5135-to-3257 : 0.5381312195121951
Crew 1, from 4757-to-1818 : 0.4144999999999999
Crew 2, from 616-to-3571 : 0.1902812195121951
Crew 3, from 3257-to-3772 : 0.04112170731707317
Crew 1, from 1818-to-1702 : 0.04888000000000001
Crew 2, from 3571-to-840 : 0.2459263414634147
Crew 3, from 3772-to-254 : 0.1896631707317073
Crew 1, from 1702-to-1060 : 0.3218017073170731
Crew 2, from 840-to-5105 : 0.4469065853658537
Crew 3, from 254-to-4661 : 0.2297153658536585
Crew 1, from 1060-to-4895 : 0.3975180487804879
Crew 2, from 5105-to-5203 : 0.2332717073170732
Crew 1, from 4895-to-3817 : 0.5060709756097563
Crew 2, from 5203-to-5062 : 0.1501724390243903
Crew 3, from 4661-to-1978 : 0.3806260975609755
Crew 2, from 5062-to-5933 : 0.1943424390243902
Crew 3, from 1978-to-6005 : 0.5715134146341464
Crew 1, from 3817-to-5698 : 0.571849024390244
Crew 2, from 5933-to-4652 : 0.2

In [50]:
pd.DataFrame({"crews": new_controls}).to_excel(
    "new_controls_"+ ds_sel +".xlsx",
    index=False
)